# Nemotron 3 Nano — Optuna Layer Search + SFT

**Workflow:**
1. Random subsample of training data with train/val split
2. Optuna searches over layer groups (attention, mamba, shared_expert) and LoRA rank
3. Each trial: apply LoRA → train → evaluate val loss → cleanup memory
4. Final training with best config (set `VAL_RATIO = 0` to retrain on all data)

**Architecture notes (from [Nemotron 3 Nano report](https://arxiv.org/abs/2512.20848)):**
- 52 layers: 6 GQA attention + 46 Mamba-2
- MoE: 128 routable experts (6 active), 2 shared experts per layer
- Router weights frozen (§3.2.5), routable experts excluded (poor LoRA ROI)
- Search space: attention, mamba, shared_expert layer groups + LoRA rank {8, 16, 32}

In [ ]:
!pip install -q --no-index --find-links /kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages datasets trl --ignore-installed

ERROR: Could not find a version that satisfies the requirement trl (from versions: none)
ERROR: No matching distribution found for trl


In [ ]:
import polars as pl
import os
import gc
import re
import torch
import torch.nn.functional as F
import mamba_ssm
import kagglehub
import optuna
from datasets import Dataset as HFDataset
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer, SFTConfig

# ── Configuration ──
SUBSAMPLE_SIZE = 400  # max is 9500
VAL_RATIO = 0.15       # set to 0.0 for final training on all data

train_full = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')
print(f"Training samples (full): {len(train_full)}")

# ── Puzzle type classifier (needed for CoT templates) ──
def classify_puzzle(prompt):
    prompt_lower = prompt.lower()
    if re.search(r'numeral system|base[- ]?\d|number.*convert|radix|secret number', prompt_lower):
        return 'Number Base Conversion'
    elif re.search(r'gravit|gravity|falling|free.?fall|acceleration due to', prompt_lower):
        return 'Gravitational Constant'
    elif re.search(r'transformation rule|equation.*transform|secret.*rule.*equation|rule.*applied.*equation', prompt_lower):
        return 'Equation Transformation'
    elif re.search(r'encrypt|cipher|secret.*code.*letter|coded.*message|secret.*text', prompt_lower):
        return 'Text Encryption'
    elif re.search(r'bit.?manipul|binary|8.?bit|bitwise|bit.*transform', prompt_lower):
        return 'Bit Manipulation'
    elif re.search(r'unit.?conver|measurement|becomes.*\d|secret.*conver.*measur', prompt_lower):
        return 'Unit Conversion'
    else:
        return 'Unknown'

train_full = train_full.with_columns(
    pl.col("prompt").map_elements(classify_puzzle, return_dtype=pl.Utf8).alias("puzzle_type")
)
print("\nPuzzle type distribution:")
print(train_full.group_by("puzzle_type").agg(pl.len().alias("count")).sort("count", descending=True))

# ── Random subsample ──
subsample = train_full.sample(n=min(SUBSAMPLE_SIZE, len(train_full)), seed=42)
print(f"\nRandom subsample: {len(subsample)} examples")

# ── Train/val split ──
if VAL_RATIO > 0:
    n_val = int(len(subsample) * VAL_RATIO)
    shuffled = subsample.sample(fraction=1.0, seed=123)
    val_df = shuffled[:n_val]
    train_df = shuffled[n_val:]
    print(f"Train: {len(train_df)}, Val: {len(val_df)}")
else:
    train_df = subsample
    val_df = None
    print(f"Train: {len(train_df)}, Val: None (VAL_RATIO=0)")

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
OUTPUT_DIR = "/kaggle/working/adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
import shutil, os, stat, sys, json
import pandas as pd

src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
shutil.copy2(src, dst)
os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

# Copy the entire bin dir so Triton can find everything
import triton.backends.nvidia as nv_backend
src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
dst_bin = "/tmp/triton_nvidia_bin"
shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
for f in os.listdir(dst_bin):
    fp = os.path.join(dst_bin, f)
    if os.path.isfile(fp):
        os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
os.environ["TRITON_PTXAS_PATH"] = dst

# Triton compiler version override (prevents ptxas version detection failures)
import triton.backends.nvidia.compiler as nv_compiler
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
nv_compiler.get_ptxas_version = lambda arch: "12.0"

# ── Bypass Triton rmsnorm EARLY (before any forward pass) ──
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

MAX_SEQ_LEN = 1024
NUM_EPOCHS = 1
GRAD_ACCUM = 8
LR = 5e-5

# ── Load model ──
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map="auto", trust_remote_code=True, dtype=torch.bfloat16,
    offload_folder="/tmp/offload",
    attn_implementation="flash_attention_2",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

# Force slow path — bypass the broken CUDA kernels
for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}: is_fast_path_available = False")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Model loaded. Vocab size: {len(tokenizer)}")

# Re-apply rmsnorm patch (model loading may import new modules)
for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn') and mod.rmsnorm_fn is not _pure_rmsnorm_fn:
        mod.rmsnorm_fn = _pure_rmsnorm_fn

# ═══════════════════════════════════════════════════════════════════════════════
# LAYER DISCOVERY — Categorize all Linear layers for Optuna search
# ═══════════════════════════════════════════════════════════════════════════════
from collections import defaultdict

layer_categories = defaultdict(list)
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        if "router" in name.lower() or "gate" in name.lower():
            layer_categories["router"].append(name)
        elif "self_attn" in name or "attention" in name:
            layer_categories["attention"].append(name)
        elif "mamba" in name.lower() or "mixer" in name.lower():
            layer_categories["mamba"].append(name)
        elif "shared_expert" in name:
            layer_categories["shared_expert"].append(name)
        elif "expert" in name:
            layer_categories["routable_expert"].append(name)
        elif "embed" in name or "lm_head" in name:
            layer_categories["embedding"].append(name)
        else:
            layer_categories["other"].append(name)

print("=" * 60)
print("LAYER CATEGORY SUMMARY")
print("=" * 60)
for cat, layers in sorted(layer_categories.items()):
    print(f"\n{cat.upper()} ({len(layers)} layers):")
    for l in layers[:5]:
        print(f"  {l}")
    if len(layers) > 5:
        print(f"  ... and {len(layers) - 5} more")

print("\nUNIQUE MODULE SUFFIXES (for target_modules):")
for cat in ["attention", "mamba", "shared_expert"]:
    if cat in layer_categories:
        suffixes = set(n.split(".")[-1] for n in layer_categories[cat])
        print(f"  {cat}: {sorted(suffixes)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DATA FORMATTING — System prompt + CoT templates + HF dataset creation
# ═══════════════════════════════════════════════════════════════════════════════

SYSTEM_PROMPT = (
    "You are a precise reasoning assistant. You will be given a puzzle that describes "
    "a hidden transformation rule with several input-output examples. Your task is to:\n"
    "1. Carefully analyze the given examples to discover the hidden rule.\n"
    "2. State the rule you discovered.\n"
    "3. Apply the rule to the given input.\n"
    "4. Provide your final answer inside \\boxed{}.\n\n"
    "Think step by step. Be concise but thorough."
)

COT_TEMPLATES = {
    'Number Base Conversion': (
        "Let me analyze the examples to find the pattern.\n\n"
        "Looking at the input-output pairs, I need to identify what numeral system "
        "conversion is being applied.\n\n"
        "After examining the examples, I can see the conversion rule.\n\n"
        "Applying this rule to the given input:\n\n"
        "\\boxed{{{answer}}}"
    ),
    'Gravitational Constant': (
        "Let me analyze the examples to find the hidden gravitational constant.\n\n"
        "From the input-output pairs, I can compute the modified value of g "
        "by comparing expected vs actual results.\n\n"
        "Applying the modified gravitational constant to the given problem:\n\n"
        "\\boxed{{{answer}}}"
    ),
    'Equation Transformation': (
        "Let me analyze the transformation rules applied to these equations.\n\n"
        "Looking at each example, I need to identify what algebraic "
        "transformations are being applied.\n\n"
        "The transformation rules I've identified are applied to the given equation:\n\n"
        "\\boxed{{{answer}}}"
    ),
    'Text Encryption': (
        "Let me analyze the encryption pattern in the examples.\n\n"
        "By comparing the input text with the encrypted output, I can identify "
        "the cipher or substitution rule being used.\n\n"
        "Applying the encryption rule to the given text:\n\n"
        "\\boxed{{{answer}}}"
    ),
    'Bit Manipulation': (
        "Let me analyze the bit transformation pattern.\n\n"
        "Converting the binary examples to decimal and examining the relationship "
        "between inputs and outputs to identify the bitwise operation.\n\n"
        "Applying the bit manipulation rule to the given binary number:\n\n"
        "\\boxed{{{answer}}}"
    ),
    'Unit Conversion': (
        "Let me analyze the secret unit conversion.\n\n"
        "Computing the ratio between output and input for each example pair "
        "to find the conversion factor.\n\n"
        "Applying the conversion factor to the given measurement:\n\n"
        "\\boxed{{{answer}}}"
    ),
}

def build_training_text(example):
    """Build chat-formatted text with system prompt + CoT reasoning."""
    prompt = example["prompt"]
    answer = str(example["answer"])
    puzzle_type = example["puzzle_type"]

    template = COT_TEMPLATES.get(puzzle_type, "After analyzing the pattern:\n\n\\boxed{{{answer}}}")
    assistant_msg = template.format(answer=answer)

    try:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": assistant_msg},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    except Exception as e:
        import warnings
        warnings.warn(
            f"apply_chat_template failed ({e}). Using manual Nemotron format."
        )
        text = (
            f"<extra_id_0>System\n{SYSTEM_PROMPT}\n"
            f"<extra_id_1>User\n{prompt}\n"
            f"<extra_id_1>Assistant\n{assistant_msg}\n"
        )
    return {"text": text}

def format_dataset(df):
    """Convert a polars DataFrame to HF dataset with chat-formatted text."""
    pdf = df.to_pandas()
    hf_ds = HFDataset.from_pandas(pdf)
    hf_ds = hf_ds.map(build_training_text, remove_columns=hf_ds.column_names)
    return hf_ds

train_dataset = format_dataset(train_df)
print(f"Train dataset: {len(train_dataset)} examples")
print(f"Example:\n{train_dataset[0]['text'][:500]}")

if val_df is not None:
    val_dataset = format_dataset(val_df)
    print(f"\nVal dataset: {len(val_dataset)} examples")
else:
    val_dataset = None
    print("\nNo validation dataset (VAL_RATIO=0)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# OPTUNA LAYER SEARCH — Find best layer groups for LoRA tuning
# Scored by validation loss (proxy for accuracy). Lower is better.
# ═══════════════════════════════════════════════════════════════════════════════

N_TRIALS = 20
SEARCH_GROUPS = ["mamba", "shared_expert"]

def build_target_modules(groups):
    """Build LoRA target module list from selected layer category names."""
    targets = []
    for g in groups:
        targets.extend(layer_categories[g])
    return targets

def make_trainer(peft_model, train_ds, val_ds=None):
    """Create SFTTrainer for a trial or final training."""
    args = SFTConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LR,
        logging_steps=5,
        bf16=True,
        max_grad_norm=1.0,
        optim="adamw_torch",
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        save_strategy="no",
        report_to="none",
        dataset_text_field="text",
        max_length=MAX_SEQ_LEN,
        packing=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": True},
    )
    return SFTTrainer(
        model=peft_model,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        args=args,
    )

def cleanup_trial(trainer_obj, peft_model_obj):
    """Aggressively free all trial memory before next trial."""
    global model
    if trainer_obj is not None:
        del trainer_obj
    if peft_model_obj is not None:
        model = peft_model_obj.unload()
        del peft_model_obj
    gc.collect()
    torch.cuda.empty_cache()

def objective(trial):
    global model

    # ── Select layer groups ──
    include_attention = trial.suggest_categorical("include_attention", [True, False])
    include_mamba = trial.suggest_categorical("include_mamba", [True, False])
    include_shared_expert = trial.suggest_categorical("include_shared_expert", [True, False])
    lora_rank = trial.suggest_categorical("lora_rank", [8, 16, 32])

    selected = []
    if include_attention:
        selected.append("attention")
    if include_mamba:
        selected.append("mamba")
    if include_shared_expert:
        selected.append("shared_expert")

    if not selected:
        return float("inf")

    target_modules = build_target_modules(selected)
    if not target_modules:
        print(f"\nTrial {trial.number}: groups={selected} yielded 0 target modules — skipping")
        return float("inf")

    print(f"\n{'='*60}")
    print(f"Trial {trial.number}: groups={selected}, rank={lora_rank}, "
          f"target_modules={len(target_modules)}")

    peft_model = None
    trainer = None
    try:
        lora_config = LoraConfig(
            r=lora_rank,
            lora_alpha=lora_rank,
            target_modules=target_modules,
            lora_dropout=0.05,
            bias="none",
            task_type=TaskType.CAUSAL_LM,
        )
        peft_model = get_peft_model(model, lora_config)
        trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
        print(f"  Trainable params: {trainable:,}")

        trainer = make_trainer(peft_model, train_dataset, val_dataset)
        trainer.train()

        if val_dataset is not None:
            eval_result = trainer.evaluate()
            score = eval_result["eval_loss"]
        else:
            score = trainer.state.log_history[-1].get("train_loss", float("inf"))

        print(f"  Score (val loss): {score:.4f}")

    except (RuntimeError, ValueError) as e:
        if "out of memory" in str(e).lower():
            print(f"  OOM error — skipping trial")
        else:
            print(f"  Trial error: {e}")
        score = float("inf")
    finally:
        cleanup_trial(trainer, peft_model)

    return score

assert val_dataset is not None, "VAL_RATIO must be > 0 for Optuna search. Set VAL_RATIO in cell 3."

study = optuna.create_study(direction="minimize", study_name="layer_search")
study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True)

print("\n" + "=" * 60)
print("OPTUNA RESULTS")
print("=" * 60)
print(f"Best trial: {study.best_trial.number}")
print(f"Best val loss: {study.best_trial.value:.4f}")
print(f"Best params: {study.best_trial.params}")

print("\nAll trials (sorted by val loss):")
for t in sorted(study.trials, key=lambda t: t.value):
    groups = [g for g in SEARCH_GROUPS if t.params.get(f"include_{g}")]
    print(f"  Trial {t.number}: loss={t.value:.4f}, rank={t.params['lora_rank']}, groups={groups}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FINAL TRAINING — Use best layers from Optuna study
# To train on ALL data: set VAL_RATIO = 0 in cell 3, re-run cells 3 + 5,
# then set BEST_GROUPS / BEST_RANK below manually and run this cell.
# ═══════════════════════════════════════════════════════════════════════════════

# Auto-populate from Optuna results, or override manually for final run
try:
    best = study.best_trial.params
    BEST_GROUPS = [g for g in SEARCH_GROUPS if best.get(f"include_{g}")]
    BEST_RANK = best["lora_rank"]
except NameError:
    BEST_GROUPS = ["attention", "mamba"]
    BEST_RANK = 32

print(f"Final training config: groups={BEST_GROUPS}, rank={BEST_RANK}")

target_modules = build_target_modules(BEST_GROUPS)
lora_config = LoraConfig(
    r=BEST_RANK,
    lora_alpha=BEST_RANK,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainer = make_trainer(model, train_dataset, val_ds=None)
print("Starting final training...")
trainer.train()

In [ ]:
import zipfile

trainer.model.save_pretrained(OUTPUT_DIR)
print(f"Adapter files saved to {OUTPUT_DIR}:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f} ({size/1024:.1f} KB)")

zip_path = "/kaggle/working/submission.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        zf.write(fpath, fname)  # files at zip root

print(f"\nCreated {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")

with zipfile.ZipFile(zip_path, 'r') as zf:
    names = zf.namelist()
    print(f"Contents: {names}")
    assert "adapter_config.json" in names, "Missing adapter_config.json!"
    print("✓ submission.zip looks correct")